# Adaptive Fusion TS2Img — Kaggle Stage 2

Runner cho Stage 2 (12 datasets × 3 seeds × 7 methods = 252 runs). Notebook hỗ trợ resume khi Kaggle giữ nguyên `stage2_resume_bundle.tar.gz` hoặc tự giải nén archive thành `adaptive_fusion_stage2/outputs` và `adaptive_fusion_stage2/cache`.


## 1. Cấu hình


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/hoangtnedu/adaptive-fusion-ts2img.git"
PROJECT_DIR = Path("/kaggle/working/adaptive-fusion-ts2img")
RUN_ROOT = Path("/kaggle/working/adaptive_fusion_stage2")
DATA_ROOT = PROJECT_DIR / "data" / "UCR"
OUTPUT_ROOT = RUN_ROOT / "outputs"
CACHE_ROOT = RUN_ROOT / "cache"

RUN_MODE = "chunk"       # "chunk" hoặc "full"
CHUNK_ID = 1             # 1, 2 hoặc 3 khi RUN_MODE="chunk"
REFRESH_REPO = True
AUTO_RESTORE = True      # Không có Input thì tự bỏ qua
RUN_TRAINING = True

if RUN_MODE not in {"chunk", "full"}:
    raise ValueError("RUN_MODE phải là 'chunk' hoặc 'full'.")
if RUN_MODE == "chunk" and CHUNK_ID not in {1, 2, 3}:
    raise ValueError("CHUNK_ID phải là 1, 2 hoặc 3.")

for path in (RUN_ROOT, OUTPUT_ROOT, CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print("Project :", PROJECT_DIR)
print("Data    :", DATA_ROOT)
print("Outputs :", OUTPUT_ROOT)
print("Cache   :", CACHE_ROOT)
print("Mode    :", RUN_MODE)
print("Chunk   :", CHUNK_ID if RUN_MODE == "chunk" else "all")


## 2. Khôi phục outputs/cache từ Kaggle Input


In [ ]:
import shutil
import tarfile

INPUT_ROOT = Path("/kaggle/input")

def count_files(path):
    return sum(1 for p in path.rglob("*") if p.is_file()) if path.exists() else 0

def score_outputs(path):
    return (
        1000 * len(list(path.rglob("summary.json")))
        + 100 * len(list(path.rglob("last_checkpoint.pt")))
        + 10 * len(list(path.rglob("best_model.pt")))
        + count_files(path)
    )

def safe_extract(bundle, destination):
    destination = destination.resolve()
    with tarfile.open(bundle, "r:gz") as tar:
        for member in tar.getmembers():
            target = (destination / member.name).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Đường dẫn không an toàn: {member.name}")
        tar.extractall(destination)

if not AUTO_RESTORE:
    print("AUTO_RESTORE=False, bỏ qua khôi phục.")
else:
    print("Các Input hiện có:")
    if INPUT_ROOT.exists():
        for item in INPUT_ROOT.iterdir():
            print(" -", item)

    restored = False
    bundles = list(INPUT_ROOT.rglob("stage2_resume_bundle.tar.gz")) if INPUT_ROOT.exists() else []

    if bundles:
        bundle = max(bundles, key=lambda p: p.stat().st_size)
        print("Restoring archive:", bundle)
        safe_extract(bundle, Path("/kaggle/working"))
        restored = True
    else:
        outputs = [p for p in INPUT_ROOT.rglob("outputs") if p.is_dir()] if INPUT_ROOT.exists() else []
        caches = [p for p in INPUT_ROOT.rglob("cache") if p.is_dir()] if INPUT_ROOT.exists() else []

        preferred = [p for p in outputs if "adaptive_fusion_stage2" in p.as_posix()]
        if preferred:
            outputs = preferred
        preferred = [p for p in caches if "adaptive_fusion_stage2" in p.as_posix()]
        if preferred:
            caches = preferred

        if outputs:
            source = max(outputs, key=score_outputs)
            shutil.copytree(source, OUTPUT_ROOT, dirs_exist_ok=True)
            print("Restored outputs from:", source)
            restored = True
        if caches:
            source = max(caches, key=count_files)
            shutil.copytree(source, CACHE_ROOT, dirs_exist_ok=True)
            print("Restored cache from:", source)
            restored = True

    print("Runs hoàn thành :", len(list(OUTPUT_ROOT.rglob("summary.json"))))
    print("Last checkpoint :", len(list(OUTPUT_ROOT.rglob("last_checkpoint.pt"))))
    print("Best checkpoint :", len(list(OUTPUT_ROOT.rglob("best_model.pt"))))
    print("Cache files      :", count_files(CACHE_ROOT))
    print("Khôi phục hoàn tất." if restored else "Không có dữ liệu cần khôi phục.")


## 3. Clone source, cài thư viện và kiểm tra GPU


In [ ]:
import os
import shutil

os.chdir("/kaggle/working")
if REFRESH_REPO and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_DIR.exists():
    !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
!python -m pip install -q -r requirements.txt
!python -m compileall src

import torch
print("Git commit:")
!git rev-parse HEAD
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn GPU accelerator trong Kaggle Settings.")
print("GPU:", torch.cuda.get_device_name(0))


## 4. Tải dữ liệu và tạo suite theo chunk


In [ ]:
import copy
import yaml

ORIGINAL_SUITE = PROJECT_DIR / "config/suites/stage2_pilot_12datasets_3seeds_all_methods.yaml"
!python -m src.cli.download_data --config {ORIGINAL_SUITE} --data-root {DATA_ROOT}

STAGE2_DATASETS = [
    "Coffee", "ECG200", "GunPoint", "FordA", "Wafer", "ItalyPowerDemand",
    "Trace", "TwoLeadECG", "CBF", "Beef", "Meat", "OliveOil",
]
CHUNKS = {
    1: ["Coffee", "ECG200", "FordA", "Beef"],
    2: ["GunPoint", "Wafer", "Trace", "Meat"],
    3: ["ItalyPowerDemand", "TwoLeadECG", "CBF", "OliveOil"],
}

missing = [
    d for d in STAGE2_DATASETS
    if not (DATA_ROOT / d / f"{d}_TRAIN.tsv").is_file()
    or not (DATA_ROOT / d / f"{d}_TEST.tsv").is_file()
]
if missing:
    raise FileNotFoundError(f"Thiếu dataset: {missing}")

with ORIGINAL_SUITE.open("r", encoding="utf-8") as f:
    base_suite = yaml.safe_load(f)

for chunk_id, datasets in CHUNKS.items():
    cfg = copy.deepcopy(base_suite)
    cfg["datasets"] = datasets
    path = PROJECT_DIR / f"config/suites/stage2_kaggle_chunk{chunk_id}.yaml"
    with path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

if RUN_MODE == "full":
    SUITE_FILE = ORIGINAL_SUITE
    SELECTED_DATASETS = STAGE2_DATASETS
else:
    SUITE_FILE = PROJECT_DIR / f"config/suites/stage2_kaggle_chunk{CHUNK_ID}.yaml"
    SELECTED_DATASETS = CHUNKS[CHUNK_ID]

EXPECTED_RUNS = len(SELECTED_DATASETS) * 3 * 7
print("Suite    :", SUITE_FILE)
print("Datasets :", SELECTED_DATASETS)
print("Expected :", EXPECTED_RUNS)


## 5. Kiểm tra tiến độ và dry run


In [ ]:
import json
from collections import Counter

records = []
for file in OUTPUT_ROOT.rglob("summary.json"):
    try:
        records.append(json.loads(file.read_text(encoding="utf-8")))
    except Exception as exc:
        print("Summary lỗi:", file, exc)

selected = [r for r in records if r.get("dataset") in SELECTED_DATASETS]
counts = Counter(r.get("dataset") for r in selected)
for dataset in SELECTED_DATASETS:
    print(f"{dataset:20s}: {counts.get(dataset, 0):2d} / 21")

SELECTED_COMPLETE = len(selected) >= EXPECTED_RUNS
print("Selected completed:", len(selected), "/", EXPECTED_RUNS)
print("Stage 2 completed :", len(records), "/ 252")
print("Last checkpoints  :", len(list(OUTPUT_ROOT.rglob("last_checkpoint.pt"))))

!python -m src.cli.batch_run   --suite {SUITE_FILE}   --set paths.data_root={DATA_ROOT} paths.output_root={OUTPUT_ROOT} paths.cache_root={CACHE_ROOT} training.num_workers=2   --dry-run


## 6. Huấn luyện hoặc tiếp tục từ checkpoint


In [ ]:
if not RUN_TRAINING:
    print("RUN_TRAINING=False, chưa chạy.")
elif SELECTED_COMPLETE:
    print("Phần đang chọn đã hoàn thành; bỏ qua huấn luyện.")
else:
    !python -m src.cli.batch_run       --suite {SUITE_FILE}       --set paths.data_root={DATA_ROOT} paths.output_root={OUTPUT_ROOT} paths.cache_root={CACHE_ROOT} training.num_workers=2


## 7. Kiểm tra kết quả, tạo paper package và resume bundle


In [ ]:
import tarfile

completed = len(list(OUTPUT_ROOT.rglob("summary.json")))
print("Completed:", completed, "/ 252")

if completed >= 252:
    PAPER_DIR = OUTPUT_ROOT / "paper_package"
    !python -m src.cli.make_paper_package       --output-root {OUTPUT_ROOT}       --results-csv {OUTPUT_ROOT}/summary_all_runs.csv       --out-dir {PAPER_DIR}       --metric test_macro_f1       --proposed adaptive_fusion_full

RESUME_ARCHIVE = Path("/kaggle/working/stage2_resume_bundle.tar.gz")
with tarfile.open(RESUME_ARCHIVE, "w:gz") as tar:
    tar.add(RUN_ROOT, arcname=RUN_ROOT.name)

print("Created:", RESUME_ARCHIVE)
print("Sau đó chọn Save Version để lưu output.")


## Cách chạy tiếp

- Chunk đang dở: giữ nguyên `CHUNK_ID`, `AUTO_RESTORE=True`, rồi chạy lại từ đầu.
- Chunk đã đủ 84 runs: chuyển `CHUNK_ID` sang phần tiếp theo.
- Không thêm `--no-resume`; source sẽ tự nạp `last_checkpoint.pt`.
